Taking in the original rates tables and trimming to our study's time period + adding a stage names column

In [49]:
import numpy as np
import pandas as pd

# Trim to 280-235 and Assign Stage Names

In [56]:
# Function to format the rates tables

from pandas import NA

def format_rates_table(df):
    df = df.copy()
    # make the age colum positive, by multiplying it by -1
    df['age'] = df['age'] * -1
    # Remove all rows where the 'age' column is > 280 or < 235
    df = df[(df['age'] <= 280) & (df['age'] >= 235)]
    # Add a new column, first column, called 'stage', empty for now
    df.insert(0, 'stage', '')
    # 'stage' should be a string that is based on the 'age' column, but with the following rules:
    df['stage'] = df['age'].apply(lambda x:
        'Kungurian'     if 274.4 < x <= 283.3 else
        'Roadian'       if 266.9 < x <= 274.4 else
        'Wordian'       if 264.28 < x <= 266.9 else
        'Capitanian'    if 259.51 < x <= 264.28 else
        'Wuchiapingian' if 254.14 < x <= 259.51 else
        'Changhsingian' if 251.902 < x <= 254.14 else
        'Induan'        if 249.9 < x <= 251.902 else
        'Olenekian'     if 246.7 < x <= 249.9 else
        'Anisian'       if 241.464 < x <= 246.7 else
        'Ladinian'      if 237 < x <= 241.464 else
        'Carnian'       if 227.3 < x <= 237 else
         ''    )
    # return the age column back to negatives
    df['age'] = df['age'] * -1
    return df

In [19]:
# Want to run this on the rates tables parent folder, 7 subfolders with 2 csvs each in them
# need to load in all csv, run function, save as new csv in same folder with _formatted at the end of the name

import os
def format_entire_folder(parent_folder):
    for subfolder in os.listdir(parent_folder):
        subfolder_path = os.path.join(parent_folder, subfolder)
        if os.path.isdir(subfolder_path):
            for file in os.listdir(subfolder_path):
                if file.endswith('.csv') and not file.endswith('_formatted.csv'):
                    file_path = os.path.join(subfolder_path, file)
                    df = pd.read_csv(file_path)
                    formatted_df = format_rates_table(df)
                    new_file_name = file.replace('.csv', '_formatted.csv')
                    new_file_path = os.path.join(subfolder_path, new_file_name)
                    formatted_df.to_csv(new_file_path, index=False)

In [20]:
# if I need to rerun the above code, I'll need to delete the _formatted csvs first

def delete_formatted_csvs(parent_folder):
    for subfolder in os.listdir(parent_folder):
        subfolder_path = os.path.join(parent_folder, subfolder)
        if os.path.isdir(subfolder_path):
            for file in os.listdir(subfolder_path):
                if file.endswith('_formatted.csv'):
                    file_path = os.path.join(subfolder_path, file)
                    os.remove(file_path)
                

In [21]:
format_entire_folder("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables")

In [17]:
delete_formatted_csvs("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables")

In [10]:
test = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_1\\rep_terr_model_1_rtt_output_table.csv")
test

,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
0,-200,0.130638,0.086023,0.177356,0.351123,0.269137,0.437642,-0.220485,-0.308983,-0.127800
1,-201,0.130638,0.086023,0.177356,0.351123,0.269137,0.437642,-0.220485,-0.308983,-0.127800
2,-202,0.130638,0.086023,0.177356,0.351123,0.269137,0.437642,-0.220485,-0.308983,-0.127800
3,-203,0.130638,0.086023,0.177356,0.351123,0.269137,0.437642,-0.220485,-0.308983,-0.127800
4,-204,0.130638,0.086023,0.177356,0.351123,0.269137,0.437642,-0.220485,-0.308983,-0.127800
...,...,...,...,...,...,...,...,...,...,...
93,-293,0.391772,0.126809,0.720715,0.143232,0.017856,0.311137,0.248540,-0.042615,0.590219
94,-294,0.391772,0.126809,0.720715,0.143232,0.017856,0.311137,0.248540,-0.042615,0.590219
95,-295,0.391772,0.126809,0.720715,0.143232,0.017856,0.311137,0.248540,-0.042615,0.590219
96,-296,0.391772,0.126809,0.720715,0.143232,0.017856,0.311137,0.248540,-0.042615,0.590219


# Remove duplicate rows in Reptilia 1Myr BDNN Runs

Not sure why there are so many additional time steps that all have identical values. Fixing that here

In [57]:
# Need to take in csvs and collapse it down so that I only keep the first and last instance of each row in a block of rows that has consecutive repeating L, M, and R values

def collapse_rows(df):
    compare_cols = df.columns[2:] # ignoring stage and age cols in the logic
    run_id = (df[compare_cols] != df[compare_cols].shift()).any(axis=1).cumsum() # boolean so it doesn't change df in place
    is_first = run_id != run_id.shift()
    is_last = run_id != run_id.shift(-1)

    return df[is_first | is_last]

In [59]:
# Only needs to run on two reptilia tables

rep_5 = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_5\\rep_all_model_5_rtt_output_table_formatted.csv")
rep_7 = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_7\\rep_terr_model_7_rtt_output_table_formatted.csv")

In [60]:
rep_5_collapsed = collapse_rows(rep_5)
# rep_5_collapsed_rounded = round_to_nearest_time_bin(rep_5_collapsed, time_bins)
rep_7_collapsed = collapse_rows(rep_7)
# rep_7_collapsed_rounded = round_to_nearest_time_bin(rep_7_collapsed, time_bins)

In [61]:
rep_5_collapsed

,stage,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
0,Carnian,-235.006031,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
16,Carnian,-235.997866,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
17,Carnian,-235.997876,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
32,Carnian,-236.989701,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
33,Carnian,-236.989711,0.051211,0.000186,0.153761,0.123410,0.002142,0.313782,-0.072199,-0.336610,0.104561
...,...,...,...,...,...,...,...,...,...,...,...
693,Kungurian,-277.902914,0.020058,0.000023,0.068683,0.036827,0.000066,0.138584,-0.016768,-0.140062,0.089707
708,Kungurian,-278.894739,0.020058,0.000023,0.068683,0.036827,0.000066,0.138584,-0.016768,-0.140062,0.089707
709,Kungurian,-278.894749,0.016303,0.000034,0.065019,0.042525,0.000132,0.149964,-0.026221,-0.159712,0.071707
724,Kungurian,-279.886574,0.016303,0.000034,0.065019,0.042525,0.000132,0.149964,-0.026221,-0.159712,0.071707


In [23]:
# save but drop index col
rep_5_collapsed.to_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_5\\rep_all_model_5_rtt_output_table_formatted_collapsed.csv", index=None)
rep_7_collapsed.to_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_7\\rep_terr_model_7_rtt_output_table_formatted_collapsed.csv", index=None)

# Collapse repeat rows

And average the ages

In [62]:
# Now need to collapse it further by doing the following: 
# every time there are two consecutive rows with the exact same values for columns 3 onwards, keep every value from the first row, except for the 'age' column, which should be an average of the two rows
# and then delete the second row. This should be done iteratively until there are no more consecutive rows with the same values for columns 3 onwards.

def collapse_consecutive_duplicates(df):
    """
    Iteratively collapses pairs of consecutive rows that have identical values
    in all columns except 'stage' and 'age'. For each collapsed pair, the
    first row's values are kept (including 'stage'), but 'age' is replaced by
    the mean of the two rows' ages.

    Iteration continues until a full pass produces no further collapsing.
    """
    compare_cols = [c for c in df.columns if c not in ('stage', 'age')]
    work = df.reset_index(drop=True).copy()

    while True:
        n = len(work)
        if n < 2:
            break

        # match[i] is True if row i and row i+1 are equal on compare_cols
        left = work[compare_cols].iloc[:-1].reset_index(drop=True)
        right = work[compare_cols].iloc[1:].reset_index(drop=True)
        match = (left == right).all(axis=1).to_numpy()

        if not match.any():
            break

        # Greedily pair from the top: if i is paired with i+1, skip i+1
        keep_mask = np.ones(n, dtype=bool)
        new_ages = work['age'].to_numpy().astype(float).copy()

        i = 0
        while i < n - 1:
            if match[i]:
                new_ages[i] = (work['age'].iat[i] + work['age'].iat[i + 1]) / 2.0
                keep_mask[i + 1] = False
                i += 2
            else:
                i += 1

        work = work.assign(age=new_ages).loc[keep_mask].reset_index(drop=True)

    return work




In [66]:
# Now take the 'age' column of the df and round it to the nearest number that it matches in a set list

time_bins = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\Time_bins_1myr_and_stages.txt", sep="\t", header=None)

def round_to_nearest_time_bin(df, time_bins):
    df = df.copy()
    time_bins = (time_bins[0].values * -1) # make time bins negative so they match the age column
    df['age'] = df['age'].apply(lambda x: min(time_bins, key=lambda y: abs(y - x)))
    return df

In [67]:
time_bins.T

,0,1,2,3,4,5,6,7,8,9,...,99,100,101,102,103,104,105,106,107,108
0,300.0,299.0,298.0,297.0,296.0,295.0,294.0,293.0,292.0,291.0,...,207.0,206.0,205.0,204.0,203.0,202.0,201.0,200.0,199.0,198.0


In [63]:
# load back in (so that the index is reset and I can do the next step of collapsing by averaging ages)
rep_5_coll = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_5\\rep_all_model_5_rtt_output_table_formatted_collapsed.csv")
rep_7_coll = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_7\\rep_terr_model_7_rtt_output_table_formatted_collapsed.csv")

In [64]:
syn_5_coll = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_5\\syn_model_5_rtt_output_table_formatted.csv")
syn_7_coll = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_7\\syn_model_7_rtt_output_table_formatted.csv")

In [65]:
rep_5_coll

,stage,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
0,Carnian,-235.006031,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
1,Carnian,-235.997866,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
2,Carnian,-235.997876,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
3,Carnian,-236.989701,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
4,Carnian,-236.989711,0.051211,0.000186,0.153761,0.123410,0.002142,0.313782,-0.072199,-0.336610,0.104561
...,...,...,...,...,...,...,...,...,...,...,...
94,Kungurian,-277.902914,0.020058,0.000023,0.068683,0.036827,0.000066,0.138584,-0.016768,-0.140062,0.089707
95,Kungurian,-278.894739,0.020058,0.000023,0.068683,0.036827,0.000066,0.138584,-0.016768,-0.140062,0.089707
96,Kungurian,-278.894749,0.016303,0.000034,0.065019,0.042525,0.000132,0.149964,-0.026221,-0.159712,0.071707
97,Kungurian,-279.886574,0.016303,0.000034,0.065019,0.042525,0.000132,0.149964,-0.026221,-0.159712,0.071707


In [68]:
rep_5_coll_rounded = round_to_nearest_time_bin(rep_5_coll, time_bins)
rep_7_coll_rounded = round_to_nearest_time_bin(rep_7_coll, time_bins)

In [70]:
rep_5_coll_rounded

,stage,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
0,Carnian,-235.0,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
1,Carnian,-236.0,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
2,Carnian,-236.0,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
3,Carnian,-237.0,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
4,Carnian,-237.0,0.051211,0.000186,0.153761,0.123410,0.002142,0.313782,-0.072199,-0.336610,0.104561
...,...,...,...,...,...,...,...,...,...,...,...
94,Kungurian,-278.0,0.020058,0.000023,0.068683,0.036827,0.000066,0.138584,-0.016768,-0.140062,0.089707
95,Kungurian,-279.0,0.020058,0.000023,0.068683,0.036827,0.000066,0.138584,-0.016768,-0.140062,0.089707
96,Kungurian,-279.0,0.016303,0.000034,0.065019,0.042525,0.000132,0.149964,-0.026221,-0.159712,0.071707
97,Kungurian,-280.0,0.016303,0.000034,0.065019,0.042525,0.000132,0.149964,-0.026221,-0.159712,0.071707


In [72]:
rep_5_coll_rounded.iloc[82:90]

,stage,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
82,Roadian,-273.0,0.049534,0.000015,0.179553,0.047411,0.000037,0.151718,0.002124,-0.170088,0.180507
83,Roadian,-274.0,0.049534,0.000015,0.179553,0.047411,0.000037,0.151718,0.002124,-0.170088,0.180507
84,Roadian,-274.0,0.042341,0.000264,0.140084,0.036641,0.000036,0.117969,0.005700,-0.121614,0.138141
85,Roadian,-274.4,0.042341,0.000264,0.140084,0.036641,0.000036,0.117969,0.005700,-0.121614,0.138141
86,Roadian,-274.4,0.040941,0.000047,0.144662,0.048069,0.000248,0.176588,-0.007128,-0.194721,0.169238
87,Kungurian,-275.0,0.040941,0.000047,0.144662,0.048069,0.000248,0.176588,-0.007128,-0.194721,0.169238
88,Kungurian,-275.0,0.034910,0.000055,0.111488,0.057939,0.000746,0.201059,-0.023029,-0.187795,0.150672
89,Kungurian,-276.0,0.034910,0.000055,0.111488,0.057939,0.000746,0.201059,-0.023029,-0.187795,0.150672


In [75]:
rep_5_collapsed_rounded_averaged =collapse_consecutive_duplicates(rep_5_coll_rounded)
rep_7_collapsed_rounded_averaged =collapse_consecutive_duplicates(rep_7_coll_rounded)
syn_5_collapsed_averaged =collapse_consecutive_duplicates(syn_5_coll)
syn_7_collapsed_averaged =collapse_consecutive_duplicates(syn_7_coll)

In [76]:
rep_5_collapsed_rounded_averaged

,stage,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
0,Carnian,-235.50,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
1,Carnian,-236.50,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
2,Carnian,-237.50,0.051211,0.000186,0.153761,0.123410,0.002142,0.313782,-0.072199,-0.336610,0.104561
3,Ladinian,-238.50,0.024237,0.000142,0.071988,0.030473,0.000227,0.089976,-0.006236,-0.081317,0.079292
4,Ladinian,-239.50,0.023190,0.000321,0.065374,0.014569,0.000082,0.042438,0.008621,-0.035354,0.065124
5,Ladinian,-240.50,0.021398,0.000176,0.060936,0.013338,0.000069,0.041207,0.008059,-0.035651,0.064897
6,Ladinian,-241.50,0.060521,0.000669,0.154743,0.037562,0.000167,0.117157,0.022959,-0.087785,0.150968
7,Anisian,-242.50,0.017794,0.000254,0.050039,0.029744,0.000133,0.075231,-0.011950,-0.070457,0.045910
8,Anisian,-243.50,0.019527,0.000282,0.054568,0.025778,0.000028,0.070229,-0.006251,-0.061441,0.052060
9,Anisian,-244.50,0.019902,0.000308,0.056788,0.036210,0.000205,0.095560,-0.016308,-0.094881,0.048432


In [77]:
# round the 'age' cols to two decimal places for all 4 dfs

rep_5_collapsed_rounded_averaged['age'] = rep_5_collapsed_rounded_averaged['age'].round(2)
rep_7_collapsed_rounded_averaged['age'] = rep_7_collapsed_rounded_averaged['age'].round(2)
syn_5_collapsed_averaged['age'] = syn_5_collapsed_averaged['age'].round(2)
syn_7_collapsed_averaged['age'] = syn_7_collapsed_averaged['age'].round(2)

## Fix 'stages'
Because the stage assignments might be messed up running collapse_consecutive_duplicates

In [81]:
# Function to format the rates tables

from pandas import NA

def format_stages(df):
    df = df.copy()
    # make the age column positive, by multiplying it by -1
    df['age'] = df['age'] * -1
    # 'stage' should be a string that is based on the 'age' column, but with the following rules:
    df['stage'] = df['age'].apply(lambda x:
        'Kungurian'     if 274.4 < x <= 283.3 else
        'Roadian'       if 266.9 < x <= 274.4 else
        'Wordian'       if 264.28 < x <= 266.9 else
        'Capitanian'    if 259.51 < x <= 264.28 else
        'Wuchiapingian' if 254.14 < x <= 259.51 else
        'Changhsingian' if 251.902 < x <= 254.14 else
        'Induan'        if 249.9 < x <= 251.902 else
        'Olenekian'     if 246.7 < x <= 249.9 else
        'Anisian'       if 241.464 < x <= 246.7 else
        'Ladinian'      if 237 < x <= 241.464 else
        'Carnian'       if 227.3 < x <= 237 else
         ''    )
        # return the age column back to negatives
    df['age'] = df['age'] * -1
    return df

In [82]:
rep_5_final = format_stages(rep_5_collapsed_rounded_averaged)
rep_7_final = format_stages(rep_7_collapsed_rounded_averaged)
syn_5_final = format_stages(syn_5_collapsed_averaged)
syn_7_final = format_stages(syn_7_collapsed_averaged)

rep_5_final

,stage,age,L_mean,L_hpd_m95,L_hpd_M95,M_mean,M_hpd_m95,M_hpd_M95,R_mean,R_hpd_m95,R_hpd_M95
0,Carnian,-235.50,0.053402,0.000204,0.142637,0.012909,0.000111,0.037128,0.040492,-0.039663,0.132992
1,Carnian,-236.50,0.046218,0.000270,0.155280,0.072098,0.002243,0.181391,-0.025880,-0.192129,0.136754
2,Ladinian,-237.50,0.051211,0.000186,0.153761,0.123410,0.002142,0.313782,-0.072199,-0.336610,0.104561
3,Ladinian,-238.50,0.024237,0.000142,0.071988,0.030473,0.000227,0.089976,-0.006236,-0.081317,0.079292
4,Ladinian,-239.50,0.023190,0.000321,0.065374,0.014569,0.000082,0.042438,0.008621,-0.035354,0.065124
5,Ladinian,-240.50,0.021398,0.000176,0.060936,0.013338,0.000069,0.041207,0.008059,-0.035651,0.064897
6,Anisian,-241.50,0.060521,0.000669,0.154743,0.037562,0.000167,0.117157,0.022959,-0.087785,0.150968
7,Anisian,-242.50,0.017794,0.000254,0.050039,0.029744,0.000133,0.075231,-0.011950,-0.070457,0.045910
8,Anisian,-243.50,0.019527,0.000282,0.054568,0.025778,0.000028,0.070229,-0.006251,-0.061441,0.052060
9,Anisian,-244.50,0.019902,0.000308,0.056788,0.036210,0.000205,0.095560,-0.016308,-0.094881,0.048432


In [86]:
# need to remove rows where age = -235 and age = -280, because those values are not consistent with the "average value of a step (time bin) in the graph", and are just artifacts of how the tables were trimmed in the very first section

rep_5_final = rep_5_final[(rep_5_final['age'] != -235) & (rep_5_final['age'] != -280)]
rep_7_final = rep_7_final[(rep_7_final['age'] != -235) & (rep_7_final['age'] != -280)]
syn_5_final = syn_5_final[(syn_5_final['age'] != -235) & (syn_5_final['age'] != -280)]
syn_7_final = syn_7_final[(syn_7_final['age'] != -235) & (syn_7_final['age'] != -280)]

In [87]:
# save all

rep_5_final.to_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_5\\rep_all_model_5_rtt_output_table_formatted_collapsed_rounded_averaged_final.csv", index=None)
rep_7_final.to_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_7\\rep_terr_model_7_rtt_output_table_formatted_collapsed_rounded_averaged_final.csv", index=None)
syn_5_final.to_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_5\\syn_model_5_rtt_output_table_formatted_collapsed_averaged_final.csv", index=None)
syn_7_final.to_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables\\model_7\\syn_model_7_rtt_output_table_formatted_collapsed_averaged_final.csv", index=None)

# Testing consistencies across rate tables

In [88]:
import os
import pandas as pd
from pathlib import Path

base = Path('C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\rates_tables\\formatted_rates_tables')

In [89]:
results = {}

for subfolder in sorted(base.iterdir()):
    if subfolder.is_dir():
        for csv_file in sorted(subfolder.glob('*.csv')):
            df = pd.read_csv(csv_file)
            results[f"{subfolder.name}/{csv_file.name}"] = len(df)

results

{'model_1/rep_terr_model_1_rtt_output_table.csv': 98,
 'model_1/rep_terr_model_1_rtt_output_table_formatted.csv': 46,
 'model_1/syn_model_1_rtt_output_table.csv': 96,
 'model_1/syn_model_1_rtt_output_table_formatted.csv': 46,
 'model_2/rep_terr_model_2_rtt_output_table.csv': 100,
 'model_2/rep_terr_model_2_rtt_output_table_formatted.csv': 46,
 'model_2/syn_models_2_and_3_rtt_output_table.csv': 100,
 'model_2/syn_models_2_and_3_rtt_output_table_formatted.csv': 47,
 'model_3/rep_all_model_3_rtt_output_table.csv': 100,
 'model_3/rep_all_model_3_rtt_output_table_formatted.csv': 46,
 'model_3/syn_models_2_and_3_rtt_output_table.csv': 100,
 'model_3/syn_models_2_and_3_rtt_output_table_formatted.csv': 47,
 'model_4/rep_all_model_4_rtt_output_table.csv': 26,
 'model_4/rep_all_model_4_rtt_output_table_formatted.csv': 14,
 'model_4/syn_model_4_rtt_output_table.csv': 26,
 'model_4/syn_model_4_rtt_output_table_formatted.csv': 14,
 'model_5/rep_all_model_5_rtt_output_table.csv': 1612,
 'model_5/rep